In [1]:
import os
import csv
from dotenv import load_dotenv 
import pandas as pd

# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.
import nest_asyncio

#create vector store chromaDB
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

#ingest document into chromaDB
from llama_index.core import Document
from llama_index.core.text_splitter import TokenTextSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.ingestion import IngestionPipeline

#setting up query engine 
from llama_index.core.prompts import PromptTemplate
from llama_index.llms.together import TogetherLLM
from llama_index.core import VectorStoreIndex

#create the evaluation dataset 
from llama_index.core.evaluation import generate_question_context_pairs
from llama_index.llms.openai import OpenAI

#evaluation metrics using GPT5mini as judge
from llama_index.core.evaluation import RetrieverEvaluator, RelevancyEvaluator, FaithfulnessEvaluator, BatchEvalRunner
from llama_index.llms.openai import OpenAI

#eval metrics using google gemini as judge
from llama_index.llms.google_genai import GoogleGenAI
import google.genai.types as types

/Users/r31881/.pyenv/versions/helloai/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/Users/r31881/.pyenv/versions/helloai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
# Load variables from .env file
load_dotenv()
nest_asyncio.apply()

In [16]:
# Access the keys
openai_key = os.getenv("OPENAI_API_KEY")
google_key = os.getenv("GOOGLE_API_KEY")
together_ai_key = os.getenv("TOGETHER_AI_API_TOKEN")

print("Keys loaded successfully")

Keys loaded successfully


### CREATE VECTOR STORE 

In [17]:
vector_store_name = "mini-llma-articles"
chroma_client = chromadb.PersistentClient(path = f"/Users/r31881/Desktop/AI/helloai/data/{vector_store_name}")
chroma_collection = chroma_client.get_or_create_collection(vector_store_name)
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)


- DOWNLOAD THE DATASET
- THIS DATASET HAS 14 ARTICLES FROM TOWARDS AI BLOG 



In [19]:
#!curl -o /Users/r31881/Desktop/AI/helloai/data/mini-llama-articles.csv https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv


- now read the data that we will store in the vector database

In [20]:
path_to_data = "/Users/r31881/Desktop/AI/helloai/data/mini-llama-articles.csv"

In [21]:
rows = []
#load the file as a json 
with open(path_to_data, mode = "r", encoding = "utf-8") as file:
    csv_reader = csv.reader(file)
    for index, row in enumerate(csv_reader):
        if index == 0:
            continue
        rows.append(row)

In [22]:
len(rows)

14

In [23]:
rows[:3]

[["Beyond GPT-4: What's New?",
  'LLM Variants and Meta\'s Open Source Before shedding light on four major trends, I\'d share the latest Meta\'s Llama 2 and Code Llama. Meta\'s Llama 2 represents a sophisticated evolution in LLMs. This suite spans models pretrained and fine-tuned across a parameter spectrum of 7 billion to 70 billion. A specialized derivative, Llama 2-Chat, has been engineered explicitly for dialogue-centric applications. Benchmarking revealed Llama 2\'s superior performance over most extant open-source chat models. Human-centric evaluations, focusing on safety and utility metrics, positioned Llama 2-Chat as a potential contender against proprietary, closed-source counterparts. The development trajectory of Llama 2 emphasized rigorous fine-tuning methodologies. Meta\'s transparent delineation of these processes aims to catalyze community-driven advancements in LLMs, underscoring a commitment to collaborative and responsible AI development. Code Llama is built on top of

### INGEST  DOCUMENTS INTO CHROMADB

- convert the data in rows list into document object so that the LLamaIndex framework can process them 


In [24]:
documents = [Document(text = row[1], metadata = {"title":row[0],"url":row[2],"source_name":row[3]}) for row in rows]

- Define the splitter object that split the text into segments with 512 tokens, 
- with a 128 overlap between the segments

In [25]:
text_splitter = TokenTextSplitter(
    separator = " ", chunk_size = 512, chunk_overlap = 128
)

- Create the pipeline to apply the transformation on each chunk. 
- and store the transformed text in the chroma vector store 
- document->chunk->embedding == node in llamaindex world
- IngestionPipeline is just a plan we have defined how this pipeline will be executed. documents->chunks->enbedding->nodes->vectorDB(chromaDB)

In [26]:
pipeline = IngestionPipeline(
    transformations = [
        text_splitter,
        HuggingFaceEmbedding(model_name = "BAAI/bge-small-en-v1.5") # or we can also use openAI embedding
    ],
    vector_store = vector_store
)

- now we will run the entire IngestionPipeline using the .run method 

In [27]:
nodes = pipeline.run(documents = documents, show_progress = True)

Parsing nodes:   0%|          | 0/14 [00:00<?, ?it/s]

Generating embeddings: 100%|██████████| 108/108 [00:01<00:00, 98.09it/s] 


In [28]:
nodes = pipeline.run(documents=documents, show_progress=True)

### SETTING UP QUERY ENGINE

- Use the Together AI service to access the Llama chat model

In [29]:
llm = TogetherLLM(
    model = "meta-llama/Llama-3.3-70B-Instruct-Turbo",   #meta-llama/Llama-4-Scout-17B-16E-Instruct
    api_key = together_ai_key
)

- create index from vector store

In [30]:
index = VectorStoreIndex.from_vector_store(vector_store, embed_model = "local:BAAI/bge-small-en-v1.5")

- define a query engine that is responsible for retrieving related pieces of text, 
- and using a LLM to formulate the final answer


In [31]:
query_engine = index.as_query_engine(llm=llm)

### TEST QUERY ENGINE 

In [32]:
res = query_engine.query("How many parameters LLaMA 2 has?")
print(res.response)
# 'Llama 2 is available in four different model sizes: 7 billion, 13 billion, 34 billion, and 70 billion parameters.'


Llama 2 is available in four different model sizes: 7 billion, 13 billion, 34 billion, and 70 billion parameters.


In [33]:
# print the source nodes used to write the answer
for src in res.source_nodes:
  print("Node ID\t", src.node_id)
  print("Title\t", src.metadata['title'])
  print("Text\t", src.text)
  print("Score\t", src.score)
  print("-_"*20)

Node ID	 ee94980c-818e-4d63-ad6a-186806895782
Title	 Meta's Llama 2: Revolutionizing Open Source Language Models for Commercial Use
Text	 I. Llama 2: Revolutionizing Commercial Use Unlike its predecessor Llama 1, which was limited to research use, Llama 2 represents a major advancement as an open-source commercial model. Businesses can now integrate Llama 2 into products to create AI-powered applications. Availability on Azure and AWS facilitates fine-tuning and adoption. However, restrictions apply to prevent exploitation. Companies with over 700 million active daily users cannot use Llama 2. Additionally, its output cannot be used to improve other language models.  II. Llama 2 Model Flavors Llama 2 is available in four different model sizes: 7 billion, 13 billion, 34 billion, and 70 billion parameters. While 7B, 13B, and 70B have already been released, the 34B model is still awaited. The pretrained variant, trained on a whopping 2 trillion tokens, boasts a context window of 4096 toke

### CREATE THE EVALUATION DATASET 

- create questions for each segment/node. These question will be used to 
- assess whether the retriever can accurately identify and return the 
- corresponding segment when queried

- in case you already have the evaljson then load it 

In [34]:
#from llama_index.finetuning.embeddings.common import EmbeddingQAFinetuneDataset
"""

from llama_index.core.evaluation import EmbeddingQAFinetuneDataset

rag_eval_dataset = EmbeddingQAFinetuneDataset.from_json(
    "./rag_eval_dataset.json"
)
"""

'\n\nfrom llama_index.core.evaluation import EmbeddingQAFinetuneDataset\n\nrag_eval_dataset = EmbeddingQAFinetuneDataset.from_json(\n    "./rag_eval_dataset.json"\n)\n'

In [35]:
llm = OpenAI(model = "gpt-5", additional_kwargs = {"reasoning_effort":"minimal"})
rag_eval_dataset = generate_question_context_pairs(
    nodes,
    llm = llm,
    num_questions_per_chunk = 1
)

100%|██████████| 108/108 [03:54<00:00,  2.18s/it]


- saving the evaluation dataset as json

In [36]:
rag_eval_dataset.save_json("/Users/r31881/Desktop/AI/helloai/data/rag_eval_dataset.json")

### Metrics to consider 

- MRR 
- HitRate 
- Faithfulness
- Relevancy


In [37]:


async def run_evaluation(index, rag_eval_dataset, top_k_values, llm_judge,llm, n_queries_to_evaluate=20,num_work=1):
    evaluation_results = {}

    # ------------------- MRR and Hit Rate -------------------

    for top_k in top_k_values:
        # Get MRR and Hit Rate
        retriever = index.as_retriever(similarity_top_k=top_k)
        retriever_evaluator = RetrieverEvaluator.from_metric_names(
            ["mrr", "hit_rate"], retriever=retriever
        )
        eval_results = await retriever_evaluator.aevaluate_dataset(rag_eval_dataset)
        avg_mrr = sum(res.metric_vals_dict["mrr"] for res in eval_results) / len(eval_results)
        avg_hit_rate = sum(res.metric_vals_dict["hit_rate"] for res in eval_results) / len(eval_results)

        # Collect the evaluation results
        evaluation_results[f"mrr_@_{top_k}"] = avg_mrr
        evaluation_results[f"hit_rate_@_{top_k}"] = avg_hit_rate

    # ------------------- Faithfulness and Relevancy -------------------

    # Extract the questions from the dataset
    queries = list(rag_eval_dataset.queries.values())
    batch_eval_queries = queries[:n_queries_to_evaluate]

    # Initiate the faithfulnes and relevancy evaluator objects
    faithfulness_evaluator = FaithfulnessEvaluator(llm=llm_judge)
    relevancy_evaluator = RelevancyEvaluator(llm=llm_judge)

    # The batch evaluator runs the evaluation in batches
    runner = BatchEvalRunner(
        {
            "faithfulness": faithfulness_evaluator,
            "relevancy": relevancy_evaluator
        },
        workers=num_work,
        show_progress=True,
    )

    # Get faithfulness and relevancy scores
    query_engine = index.as_query_engine(llm=llm)
    eval_results = await runner.aevaluate_queries(
        query_engine, queries=batch_eval_queries
    )
    faithfulness_score = sum(result.passing for result in eval_results['faithfulness']) / len(eval_results['faithfulness'])
    relevancy_score = sum(result.passing for result in eval_results['relevancy']) / len(eval_results['relevancy'])
    evaluation_results["faithfulness"] = faithfulness_score
    evaluation_results["relevancy"] = relevancy_score

    return evaluation_results

- running evaluations

In [36]:
# We evaluate the retrievers with different top_k values.
top_k_values = [2, 4, 6, 8, 10]

llm_judge = OpenAI(model="gpt-5",additional_kwargs={"reasoning_effort":"minimal"})

evaluation_results = await run_evaluation(index, rag_eval_dataset, top_k_values, llm_judge,llm=llm,n_queries_to_evaluate=20,num_work=1)

100%|██████████| 40/40 [00:40<00:00,  1.02s/it]


In [37]:
print(evaluation_results)

{'mrr_@_2': 0.8287037037037037, 'hit_rate_@_2': 0.9074074074074074, 'mrr_@_4': 0.8495370370370371, 'hit_rate_@_4': 0.9722222222222222, 'mrr_@_6': 0.8513888888888889, 'hit_rate_@_6': 0.9814814814814815, 'mrr_@_8': 0.8513888888888889, 'hit_rate_@_8': 0.9814814814814815, 'mrr_@_10': 0.8513888888888889, 'hit_rate_@_10': 0.9814814814814815, 'faithfulness': 1.0, 'relevancy': 1.0}


- trying google gemini as judge 

In [40]:
from llama_index.llms.google_genai import GoogleGenAI

 # Use Gemini as the LLM model
import google.genai.types as types

config = types.GenerateContentConfig(thinking_config=types.ThinkingConfig(thinking_budget=0))
llm = GoogleGenAI(model="meta-llama/Llama-3.3-70B-Instruct-Turbo", generation_config=config)

# run evaluation with Gemini
top_k_values = [2, 4, 6, 8, 10]
llm_judge = OpenAI(temperature=0, model="gpt-5",additional_kwargs={"reasoning_effort":"minimal"})
evaluation_results = await run_evaluation(index, rag_eval_dataset, top_k_values, llm_judge,llm=llm, n_queries_to_evaluate=20,num_work=1)

ClientError: 404 Not Found. {'message': '', 'status': 'Not Found'}

### INFERENCE SPEED COMPARISON 

In [38]:
import time

In [42]:
llm = TogetherLLM(
    model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
    api_key=os.environ["TOGETHER_AI_API_TOKEN"]
)

time_start = time.time()
llm.complete("List the 50 states in the United States of America. Write their names in a comma-separated list and nothing else.")
time_end = time.time()
print("Time taken for Llama 4 Scout on Together AI: {0:.2f} seconds".format(time_end - time_start))

Time taken for Llama 4 Scout on Together AI: 1.11 seconds


In [43]:
llm = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort":"minimal"})

time_start = time.time()
llm.complete("List the 50 states in the United States of America. Write their names in a comma-separated list and nothing else.")
time_end = time.time()
print("Time taken for GPT 5 Mini: {0:.2f} seconds".format(time_end - time_start))

Time taken for GPT 5 Mini: 2.50 seconds


In [44]:
from llama_index.llms.google_genai import GoogleGenAI
import google.genai.types as types

config = types.GenerateContentConfig(thinking_config=types.ThinkingConfig(thinking_budget=0))

llm = GoogleGenAI(model="gemini-3-flash-preview", generation_config=config)

time_start = time.time()
llm.complete("List the 50 states in the United States of America. Write their names in a comma-separated list and nothing else.")
time_end = time.time()
print("Time taken for Gemini 2.5 Flash: {0:.2f} seconds".format(time_end - time_start))

Time taken for Gemini 2.5 Flash: 14.49 seconds
